# Chapter 11: God Spiked the Integers

Chapter 10 introduced the GLM framework and basic binomial/Poisson models. Chapter 11 goes deeper:

| Topic | New idea |
|-------|----------|
| Chimpanzee experiment | Disaggregated binomial with multiple index variables (actor + treatment) |
| Aggregated vs disaggregated | Same likelihood, two data shapes |
| Poisson with offset | Exposure/observation window differs across units |
| Zero-inflated Poisson | Mixture model — two processes generating zeros |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from scipy import stats, special

plt.style.use('default')
%matplotlib inline
rng = np.random.default_rng(42)

print(f'PyMC {pm.__version__} | ArviZ {az.__version__}')

## Part 1: Chimpanzee Prosocial Experiment

### The Setup

7 chimpanzees complete a series of trials. On each trial, the chimp can pull a left or right lever. One lever is *prosocial* (gives food to both the chimp AND a partner), the other is *selfish* (gives food only to the chimp).

**Key variables**:
- `pulled_left`: did the chimp pull left? (0/1) — the outcome
- `prosoc_left`: was the prosocial option on the left? (0/1)
- `condition`: was a partner present? (0=alone, 1=partner present)
- `actor`: which chimp (1–7)
- `block`: which experimental block (1–6)

**Scientific question**: Do chimps pull the prosocial lever more often when a partner is present? (Do they care about others?)

### The Model Design

Rather than `β_prosoc × prosoc_left + β_condition × condition + β_interaction × prosoc_left × condition`, McElreath uses an **index variable** for the 4 treatment combinations:

| treatment | prosoc_left | condition (partner) |
|-----------|-------------|---------------------|
| 1 | 0 | 0 |
| 2 | 1 | 0 |
| 3 | 0 | 1 |
| 4 | 1 | 1 |

Model:
$$\text{pulled\_left}_i \sim \text{Bernoulli}(p_i)$$
$$\text{logit}(p_i) = \alpha_{\text{actor}[i]} + \beta_{\text{treatment}[i]}$$

Each chimp gets its own baseline tendency (`α[actor]`). Each treatment combination gets its own effect (`β[treatment]`). **No interaction term needed** — the index variable encodes all combinations directly.

In [ ]:
url = 'https://raw.githubusercontent.com/rmcelreath/rethinking/master/data/chimpanzees.csv'
chimps = pd.read_csv(url, sep=';')
print(chimps.head(10))
print(f'\nShape: {chimps.shape}')
print(f'Actors: {sorted(chimps["actor"].unique())}')
print(f'Blocks: {sorted(chimps["block"].unique())}')

In [ ]:
# Build treatment index (1-indexed → 0-indexed)
chimps['treatment'] = (
    chimps['prosoc_left'] + 2 * chimps['condition']
).astype(int)  # 0,1,2,3

treatment_labels = [
    'R/no partner',   # prosoc_left=0, condition=0
    'L/no partner',   # prosoc_left=1, condition=0
    'R/partner',      # prosoc_left=0, condition=1
    'L/partner',      # prosoc_left=1, condition=1
]

pulled_left = chimps['pulled_left'].values
actor       = chimps['actor'].values - 1        # 0-indexed
treatment   = chimps['treatment'].values         # 0-indexed
block_id    = chimps['block'].values - 1         # 0-indexed

n_actors     = chimps['actor'].nunique()
n_treatments = chimps['treatment'].nunique()
n_blocks     = chimps['block'].nunique()

print(f'N = {len(pulled_left)}, actors = {n_actors}, treatments = {n_treatments}, blocks = {n_blocks}')

# Quick summary: pull rate by actor and treatment
summary = chimps.groupby(['actor', 'treatment'])['pulled_left'].mean().unstack()
summary.columns = treatment_labels
print('\nProportion pulling left, by actor × treatment:')
print(summary.round(2).to_string())

In [ ]:
# Quick visualisation: pull rates by actor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: by actor
ax = axes[0]
actor_rates = chimps.groupby('actor')['pulled_left'].mean()
colors = plt.cm.Set2(np.linspace(0, 1, n_actors))
bars = ax.bar(actor_rates.index, actor_rates.values, color=colors, edgecolor='white')
ax.axhline(0.5, color='gray', ls='--', lw=1.5, alpha=0.7, label='50% (no preference)')
ax.set_xlabel('Actor (chimp)', fontsize=11)
ax.set_ylabel('P(pull left)', fontsize=11)
ax.set_title('Pull-left rate by chimp\n(huge individual variation!)', fontsize=11, fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, 1)

# Panel 2: by treatment
ax = axes[1]
treat_rates = chimps.groupby('treatment')['pulled_left'].mean()
treat_colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
ax.bar(range(n_treatments), treat_rates.values, color=treat_colors, edgecolor='white')
ax.axhline(0.5, color='gray', ls='--', lw=1.5, alpha=0.7)
ax.set_xticks(range(n_treatments))
ax.set_xticklabels(treatment_labels, fontsize=9)
ax.set_ylabel('P(pull left)', fontsize=11)
ax.set_title('Pull-left rate by treatment\n(L options pulled more — makes sense!)', fontsize=11, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, 1)

plt.suptitle('Chimpanzee Prosocial Experiment — Raw Rates', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Prior Predictive Check

Priors on the logit scale:
- `α[actor] ~ Normal(0, 1.5)` — individual baseline tendency, covers most of the probability range
- `β[treatment] ~ Normal(0, 0.5)` — treatment effects; smaller prior because we expect them to be modest relative to individual differences

In [ ]:
# Prior predictive: what do these priors imply for p?
n_sim = 1000
alpha_sim = rng.normal(0, 1.5, n_sim)
beta_sim  = rng.normal(0, 0.5, n_sim)
p_sim = special.expit(alpha_sim + beta_sim)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(p_sim, bins=50, color='steelblue', alpha=0.75, edgecolor='white', density=True)
ax.set_xlabel('Prior predicted p (pull left)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Prior predictive: α~N(0,1.5) + β~N(0,0.5) → probability scale\n'
             'Covers the full range but not strongly U-shaped', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Prior P(pull left) — mean: {p_sim.mean():.2f},  std: {p_sim.std():.2f}')
print(f'                    5th–95th pct: [{np.percentile(p_sim,5):.2f}, {np.percentile(p_sim,95):.2f}]')

### Fit the Model

In [ ]:
coords = {
    'actor':     [f'chimp_{i+1}' for i in range(n_actors)],
    'treatment': treatment_labels,
}

with pm.Model(coords=coords) as m_chimps:
    # Priors
    alpha = pm.Normal('alpha', mu=0, sigma=1.5, dims='actor')
    beta  = pm.Normal('beta',  mu=0, sigma=0.5, dims='treatment')

    # Linear model on logit scale
    logit_p = alpha[actor] + beta[treatment]
    p = pm.Deterministic('p', pm.math.invlogit(logit_p))

    # Likelihood (disaggregated — individual trial level)
    obs = pm.Bernoulli('obs', p=p, observed=pulled_left)

    idata_chimps = pm.sample(1000, tune=1000, chains=4,
                             target_accept=0.9, random_seed=42, progressbar=True)
    pm.compute_log_likelihood(idata_chimps)

print(az.summary(idata_chimps, var_names=['alpha', 'beta'], hdi_prob=0.89).to_string())

In [ ]:
# Forest plot: actor intercepts and treatment effects
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

az.plot_forest(idata_chimps, var_names=['alpha'], hdi_prob=0.89,
               combined=True, ax=axes[0])
axes[0].axvline(0, color='red', ls='--', lw=1.2, alpha=0.6)
axes[0].set_title('Actor intercepts α\n(log-odds scale)', fontsize=11, fontweight='bold')
axes[0].grid(True, axis='x', alpha=0.3)

az.plot_forest(idata_chimps, var_names=['beta'], hdi_prob=0.89,
               combined=True, ax=axes[1])
axes[1].axvline(0, color='red', ls='--', lw=1.2, alpha=0.6)
axes[1].set_title('Treatment effects β\n(log-odds scale)', fontsize=11, fontweight='bold')
axes[1].grid(True, axis='x', alpha=0.3)

plt.suptitle('Chimpanzee Model Posteriors (89% HDI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Key question: does having a partner (condition=1) increase prosocial pulling?
# Compare treatments 1 vs 3 (right option) and 2 vs 4 (left option)
post = idata_chimps.posterior
beta_s = post['beta'].values.reshape(-1, 4)  # (S, 4): R/no, L/no, R/yes, L/yes

# Contrast: partner present vs absent, averaged over left/right
# Treatment 2 (L/partner) - Treatment 1 (L/no partner)
# Treatment 3 (R/partner) - Treatment 0 (R/no partner)
contrast_L = beta_s[:, 3] - beta_s[:, 1]   # L/partner - L/no partner
contrast_R = beta_s[:, 2] - beta_s[:, 0]   # R/partner - R/no partner
contrast_avg = (contrast_L + contrast_R) / 2

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, contrast, title, color in zip(axes,
    [contrast_L, contrast_R, contrast_avg],
    ['Left option:\npartner vs no partner', 'Right option:\npartner vs no partner',
     'Average partner\neffect'],
    ['steelblue', 'tomato', 'darkorchid']):
    az.plot_posterior({'contrast': contrast}, hdi_prob=0.89,
                      ref_val=0, ax=ax, color=color)
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.suptitle('Does having a partner increase prosocial pulling?\n'
             '(positive = more prosocial when partner present)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Average partner effect (log-odds):')
print(f'  Mean = {contrast_avg.mean():+.3f}')
lo, hi = np.percentile(contrast_avg, [5.5, 94.5])
print(f'  89% CI = [{lo:+.3f}, {hi:+.3f}]')
print(f'  P(partner increases prosocial) = {(contrast_avg > 0).mean():.1%}')
print()
print('Conclusion: little evidence chimps pull prosocially more when partner is present.')
print('Individual actor differences swamp the treatment effect.')

## Part 2: Aggregated vs Disaggregated Binomial

The same likelihood can be expressed two ways depending on how data is structured:

| Format | Outcome | Likelihood |
|--------|---------|------------|
| **Disaggregated** | 0/1 per trial (1504 rows) | `Bernoulli(p)` |
| **Aggregated** | count of successes out of N trials (28 rows) | `Binomial(N, p)` |

Mathematically identical — same log-likelihood, same posterior. Aggregation just sums multiple Bernoullis.

In [ ]:
# Aggregate: count pulls per actor × treatment
agg = (chimps.groupby(['actor', 'treatment'])
             .agg(pulled=('pulled_left', 'sum'),
                  total=('pulled_left', 'count'))
             .reset_index())

actor_agg     = agg['actor'].values - 1
treatment_agg = agg['treatment'].values
pulled_agg    = agg['pulled'].values
total_agg     = agg['total'].values

print(f'Aggregated data: {len(agg)} rows (28 actor×treatment combinations)')
print(agg.head(8).to_string(index=False))

with pm.Model(coords=coords) as m_chimps_agg:
    alpha = pm.Normal('alpha', mu=0, sigma=1.5, dims='actor')
    beta  = pm.Normal('beta',  mu=0, sigma=0.5, dims='treatment')
    logit_p = alpha[actor_agg] + beta[treatment_agg]
    p = pm.Deterministic('p', pm.math.invlogit(logit_p))
    # Binomial instead of Bernoulli — counts successes out of total
    obs = pm.Binomial('obs', n=total_agg, p=p, observed=pulled_agg)

    idata_agg = pm.sample(1000, tune=1000, chains=4,
                          target_accept=0.9, random_seed=42, progressbar=True)

# Compare posteriors — should be essentially identical
summ_disagg = az.summary(idata_chimps, var_names=['beta'], hdi_prob=0.89)[['mean','sd']]
summ_agg    = az.summary(idata_agg,    var_names=['beta'], hdi_prob=0.89)[['mean','sd']]

print('\nβ posteriors — disaggregated (Bernoulli) vs aggregated (Binomial):')
comparison = pd.concat([summ_disagg.add_suffix('_disagg'),
                        summ_agg.add_suffix('_agg')], axis=1)
print(comparison.round(4).to_string())
print('\n✓ Posteriors are identical — same likelihood, different data shape.')

## Part 3: Poisson with Offset (Exposure)

### The Problem

Sometimes observation windows differ. A monastery records manuscripts per day, but some months have more monks than others, or one monastery was only observed for half the period. The *rate* (manuscripts per monk per day) is what we want — but raw counts depend on exposure.

### The Solution: Offset

$$y_i \sim \text{Poisson}(\lambda_i)$$
$$\log(\lambda_i) = \log(\text{exposure}_i) + \alpha + \beta x_i$$

The `log(exposure)` term is an **offset** — a predictor with coefficient fixed to 1. It converts the model from predicting *counts* to predicting *rates*:
$$\lambda_i = \text{exposure}_i \cdot e^{\alpha + \beta x_i}$$

So `e^α` is the baseline *rate* (per unit exposure), not the baseline count.

In [ ]:
# Simulate monastery data
# Two monasteries, observed for different numbers of days
# True rate: monastery A = 1.5 manuscripts/day, B = 1.0/day
rng_data = np.random.default_rng(7)

n_days_A = 30
n_days_B = 36
rate_A   = 1.5
rate_B   = 1.0

y_A = rng_data.poisson(rate_A, n_days_A)
y_B = rng_data.poisson(rate_B, n_days_B)

monastery = pd.DataFrame({
    'manuscripts': np.concatenate([y_A, y_B]),
    'days':        np.concatenate([np.ones(n_days_A), np.ones(n_days_B)]),
    'monastery':   np.concatenate([np.zeros(n_days_A, int), np.ones(n_days_B, int)]),
})

print(f'Monastery A: {n_days_A} days, {y_A.sum()} total manuscripts, raw rate = {y_A.mean():.2f}/day')
print(f'Monastery B: {n_days_B} days, {y_B.sum()} total manuscripts, raw rate = {y_B.mean():.2f}/day')
print(f'\nTrue rates: A={rate_A}, B={rate_B}')

y          = monastery['manuscripts'].values
log_days   = np.log(monastery['days'].values)  # offset (all 1 day here, so log=0)
mon_id     = monastery['monastery'].values

# Fit with offset
with pm.Model() as m_monastery:
    # alpha[monastery] = log(rate) per day
    alpha = pm.Normal('alpha', mu=0, sigma=1, shape=2)
    # log(lambda) = log(days) + alpha[monastery]  ← offset fixes coeff to 1
    log_lam = log_days + alpha[mon_id]
    lam = pm.Deterministic('lambda', pm.math.exp(log_lam))
    obs = pm.Poisson('obs', mu=lam, observed=y)

    idata_mon = pm.sample(1000, tune=1000, chains=4,
                          target_accept=0.9, random_seed=42, progressbar=True)

# Recover rates
post_mon = idata_mon.posterior
alpha_s  = post_mon['alpha'].values.reshape(-1, 2)
rate_A_post = np.exp(alpha_s[:, 0])
rate_B_post = np.exp(alpha_s[:, 1])

print(f'\nPosterior rate estimates (manuscripts/day):')
print(f'  Monastery A: {rate_A_post.mean():.2f}  89% CI [{np.percentile(rate_A_post,5.5):.2f}, {np.percentile(rate_A_post,94.5):.2f}]  (true: {rate_A})')
print(f'  Monastery B: {rate_B_post.mean():.2f}  89% CI [{np.percentile(rate_B_post,5.5):.2f}, {np.percentile(rate_B_post,94.5):.2f}]  (true: {rate_B})')

## Part 4: Zero-Inflated Poisson (ZIP)

### The Problem

Sometimes count data has **more zeros than a Poisson can produce**. Example: number of manuscripts produced per day by monks, but on some days monks take a break from writing entirely (drinking wine, apparently). There are two processes generating zeros:

1. **Drinking process**: with probability `ψ`, monks drink instead of work → always 0 manuscripts
2. **Working process**: with probability `1 - ψ`, monks work → Poisson(λ) manuscripts (which could also be 0)

### The Model

$$P(y_i = 0) = \psi + (1-\psi)\cdot e^{-\lambda}$$
$$P(y_i = k > 0) = (1-\psi)\cdot \text{Poisson}(k \mid \lambda)$$

Two parameters to estimate:
- `ψ` (psi): probability of the zero-generating process (drinking)
- `λ` (lambda): rate when actually working

In [ ]:
# Simulate ZIP data
true_psi    = 0.20   # 20% of days monks drink
true_lambda = 1.5    # manuscripts per working day
N = 365

drinking = rng.binomial(1, true_psi, N)          # 1 = drinking day
y_zip    = rng.poisson(true_lambda, N) * (1 - drinking)  # 0 if drinking

print(f'True ψ = {true_psi}, true λ = {true_lambda}')
print(f'Observed zeros: {(y_zip == 0).sum()} / {N} = {(y_zip==0).mean():.1%}')
print(f'Expected zeros from Poisson alone: {np.exp(-true_lambda):.1%}')
print(f'Extra zeros from drinking: ~{true_psi:.0%}')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(y_zip, bins=range(0, y_zip.max()+2), color='steelblue', alpha=0.75,
        edgecolor='white', density=True, align='left')
# Overlay pure Poisson for comparison
k = np.arange(0, y_zip.max()+1)
ax.plot(k, stats.poisson.pmf(k, true_lambda), 'r--o', ms=5, lw=2,
        label=f'Pure Poisson(λ={true_lambda})')
ax.set_xlabel('Manuscripts per day', fontsize=11)
ax.set_ylabel('Proportion', fontsize=11)
ax.set_title('Zero-inflated data:\nmore zeros than Poisson predicts', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(['Zeros (actual)', f'Zeros\n(Poisson only)'],
       [(y_zip==0).mean(), np.exp(-true_lambda)],
       color=['steelblue', 'tomato'], alpha=0.8, edgecolor='white')
ax.set_ylabel('Proportion of days', fontsize=11)
ax.set_title('Excess zeros\n(the signature of zero-inflation)', fontsize=11, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Zero-Inflated Poisson: Two Processes Generating Zeros', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Fit ZIP model in PyMC
with pm.Model() as m_zip:
    # psi: probability of drinking (zero-inflation)
    psi = pm.Beta('psi', alpha=2, beta=8)    # weakly informative: most days work

    # lambda: manuscript rate on working days
    lam = pm.Exponential('lambda', lam=1)

    # ZeroInflatedPoisson is built into PyMC
    obs = pm.ZeroInflatedPoisson('obs', psi=psi, mu=lam, observed=y_zip)

    idata_zip = pm.sample(1000, tune=1000, chains=4,
                          target_accept=0.9, random_seed=42, progressbar=True)

print(az.summary(idata_zip, hdi_prob=0.89).to_string())
print()

psi_post = idata_zip.posterior['psi'].values.flatten()
lam_post = idata_zip.posterior['lambda'].values.flatten()
print(f'True ψ={true_psi:.2f}, estimated ψ={psi_post.mean():.3f}  '
      f'89% CI [{np.percentile(psi_post,5.5):.3f}, {np.percentile(psi_post,94.5):.3f}]')
print(f'True λ={true_lambda:.2f}, estimated λ={lam_post.mean():.3f}  '
      f'89% CI [{np.percentile(lam_post,5.5):.3f}, {np.percentile(lam_post,94.5):.3f}]')

In [ ]:
# Compare: naive Poisson vs ZIP
with pm.Model() as m_poisson_naive:
    lam_naive = pm.Exponential('lambda', lam=1)
    obs = pm.Poisson('obs', mu=lam_naive, observed=y_zip)
    idata_naive = pm.sample(1000, tune=1000, chains=4,
                            target_accept=0.9, random_seed=42, progressbar=True)
    pm.compute_log_likelihood(idata_naive)

with m_zip:
    pm.compute_log_likelihood(idata_zip)

comparison = az.compare(
    {'ZIP': idata_zip, 'naive Poisson': idata_naive},
    ic='waic', scale='deviance'
)
print('WAIC comparison — ZIP vs naive Poisson:')
print(comparison.to_string())

lam_naive_post = idata_naive.posterior['lambda'].values.flatten()
print(f'\nNaive Poisson λ = {lam_naive_post.mean():.3f}  (biased — absorbs drinking days into lower rate)')
print(f'ZIP         λ = {lam_post.mean():.3f}  (correctly separates working rate from drinking probability)')

## Key Insights

### 1. Index variables beat interaction terms for categorical combinations
Instead of `β_prosoc × x1 + β_condition × x2 + β_interaction × x1 × x2`, define a single index over all {treatment} combinations. Cleaner model, easier to set priors independently per combination, and the contrasts you care about are explicit.

### 2. Aggregated and disaggregated binomial are the same model
- Disaggregated: `Bernoulli(p)` on individual 0/1 rows
- Aggregated: `Binomial(N, p)` on summed counts

Same likelihood, same posterior. Choose the format that matches your data. Aggregated is faster when N is large.

### 3. Individual variation can swamp treatment effects
The chimpanzee model shows that actor (individual) differences in baseline pull tendency are enormous — the treatment (partner presence) effect is small and uncertain by comparison. This is why controlling for individual variation matters: a model without `α[actor]` would give biased treatment estimates.

### 4. Offset converts counts to rates
When observation windows differ, add `log(exposure)` as an offset with coefficient fixed to 1. This makes the model estimate a **rate** (events per unit exposure) rather than a raw count. Without offset, longer-observed units would appear to have higher rates.

### 5. Zero-inflated models separate two processes
When you have excess zeros, ask: are the zeros all from the same process, or from two different processes? ZIP says:
- Process 1 (with prob ψ): always produces a zero
- Process 2 (with prob 1−ψ): produces Poisson(λ) counts, including possibly zero

A naive Poisson fit to ZIP data will estimate a lower λ than the true working rate, because it conflates drinking-zeros and working-zeros.

---

*Chapter 11 — Statistical Rethinking (McElreath, 2nd ed.)*